# Data Proyeksi Iklim — Studi Kasus: Yogyakarta
### Eksplorasi Data Historis dan GCM CMIP6
**Studi Kasus:** Suhu Harian dan Curah Hujan Kota Yogyakarta  
**Data:** Observasi historis (1990–2020) + GCM SSP2-4.5 & SSP5-8.5 (2021–2060)

---
**Alur:**
```
1. Generate demo data (Obs historis + GCM dua skenario)
2. Statistik deskriptif
3. Visualisasi time series
4. Siklus musiman (climatology)
5. Perbandingan periode historis vs masa depan
6. Export CSV
```

## 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('✅ Library berhasil diimport')

---
## 1. Generate Demo Data

> **Catatan:** Dalam penelitian nyata, data ini diganti dengan:
> - **Observasi** → data stasiun BMKG atau ERA5 reanalysis
> - **GCM** → download dari CMIP6 via [ESGF Portal](https://esgf-node.llnl.gov/search/cmip6/)
> - Skenario yang umum digunakan: **SSP2-4.5** (emisi menengah) dan **SSP5-8.5** (emisi tinggi)

In [ ]:
# ============================================================
# PERIODE DATA
# ============================================================
dates_hist = pd.date_range('1990-01-01', '2020-12-31', freq='D')
dates_fut  = pd.date_range('2021-01-01', '2060-12-31', freq='D')
n_hist = len(dates_hist)
n_fut  = len(dates_fut)

print(f'Historis : {n_hist:,} hari ({dates_hist[0].date()} s/d {dates_hist[-1].date()})')
print(f'Future   : {n_fut:,} hari ({dates_fut[0].date()} s/d {dates_fut[-1].date()})')

In [ ]:
# ============================================================
# FUNGSI HELPER: siklus musiman
# ============================================================
def musiman_suhu(dates, base=27.5, amp=2.0):
    """Siklus musiman suhu harian (lebih panas di bulan kering)."""
    doy = dates.day_of_year
    return base + amp * np.sin(2 * np.pi * (doy - 80) / 365)

def musiman_hujan(dates, base=6.0, amp=5.0):
    """Siklus musiman curah hujan harian (wet/dry season Indonesia)."""
    doy = dates.day_of_year
    # Puncak hujan sekitar Jan-Mar (DOY 1-90)
    return np.clip(base - amp * np.sin(2 * np.pi * (doy - 10) / 365), 0.5, None)

# ============================================================
# 1A. DATA HISTORIS (Obs / Reanalysis)
# ============================================================
suhu_base_hist = musiman_suhu(dates_hist)
hujan_base_hist = musiman_hujan(dates_hist)

obs = pd.DataFrame({
    'date'  : dates_hist,
    'suhu'  : suhu_base_hist + np.random.normal(0, 1.5, n_hist),
    'hujan' : np.random.exponential(hujan_base_hist),
})
obs['hujan'] = obs['hujan'].clip(lower=0)

print('Data Historis (Obs):')
print(obs[['suhu','hujan']].describe().round(2))

In [ ]:
# ============================================================
# 1B. GCM SSP2-4.5 (skenario emisi menengah)
# Pemanasan ~+1.5°C di akhir periode 2060
# ============================================================
suhu_base_fut = musiman_suhu(dates_fut)
hujan_base_fut = musiman_hujan(dates_fut)
warming_245 = np.linspace(0, 1.5, n_fut)   # tren +1.5°C selama 40 tahun

gcm_245 = pd.DataFrame({
    'date'  : dates_fut,
    'suhu'  : suhu_base_fut + warming_245 + np.random.normal(0, 1.6, n_fut),
    'hujan' : np.random.exponential(hujan_base_fut * (1 + 0.1 * warming_245 / 1.5)),
})
gcm_245['hujan'] = gcm_245['hujan'].clip(lower=0)

# ============================================================
# 1C. GCM SSP5-8.5 (skenario emisi tinggi)
# Pemanasan ~+3.0°C di akhir periode 2060
# ============================================================
warming_585 = np.linspace(0, 3.0, n_fut)   # tren +3.0°C selama 40 tahun

gcm_585 = pd.DataFrame({
    'date'  : dates_fut,
    'suhu'  : suhu_base_fut + warming_585 + np.random.normal(0, 1.8, n_fut),
    'hujan' : np.random.exponential(hujan_base_fut * (1 + 0.2 * warming_585 / 3.0)),
})
gcm_585['hujan'] = gcm_585['hujan'].clip(lower=0)

print('GCM SSP2-4.5:')
print(gcm_245[['suhu','hujan']].describe().round(2))
print('\nGCM SSP5-8.5:')
print(gcm_585[['suhu','hujan']].describe().round(2))

---
## 2. Statistik Deskriptif

In [ ]:
# ============================================================
# TABEL PERBANDINGAN STATISTIK
# ============================================================
def ringkasan(df, label):
    return pd.Series({
        'Dataset'   : label,
        'Suhu Mean (°C)' : df['suhu'].mean(),
        'Suhu Std (°C)'  : df['suhu'].std(),
        'Suhu Min (°C)'  : df['suhu'].min(),
        'Suhu Max (°C)'  : df['suhu'].max(),
        'Hujan Mean (mm)': df['hujan'].mean(),
        'Hujan Max (mm)' : df['hujan'].max(),
    })

tabel = pd.DataFrame([
    ringkasan(obs,     'Historis (1990–2020)'),
    ringkasan(gcm_245, 'SSP2-4.5 (2021–2060)'),
    ringkasan(gcm_585, 'SSP5-8.5 (2021–2060)'),
]).set_index('Dataset')

print('📊 Statistik Deskriptif Perbandingan:')
print(tabel.round(2).to_string())

---
## 3. Visualisasi Time Series

In [ ]:
# ============================================================
# PLOT: Time series suhu tahunan
# ============================================================
def rata_tahunan(df, col):
    df = df.copy()
    df['year'] = df['date'].dt.year
    return df.groupby('year')[col].mean()

suhu_hist_yr  = rata_tahunan(obs, 'suhu')
suhu_245_yr   = rata_tahunan(gcm_245, 'suhu')
suhu_585_yr   = rata_tahunan(gcm_585, 'suhu')
hujan_hist_yr = rata_tahunan(obs, 'hujan') * 365  # konversi ke mm/tahun
hujan_245_yr  = rata_tahunan(gcm_245, 'hujan') * 365
hujan_585_yr  = rata_tahunan(gcm_585, 'hujan') * 365

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# --- Suhu ---
ax = axes[0]
ax.plot(suhu_hist_yr.index, suhu_hist_yr, color='tomato', lw=2,
        marker='o', ms=3, label='Historis (Obs)')
ax.plot(suhu_245_yr.index, suhu_245_yr, color='steelblue', lw=2, label='GCM SSP2-4.5')
ax.plot(suhu_585_yr.index, suhu_585_yr, color='darkorange', lw=2, label='GCM SSP5-8.5')
ax.axvline(2020.5, color='gray', lw=1.5, linestyle='--', alpha=0.7)
ax.text(2021, ax.get_ylim()[0]+0.05, '← Historis | Future →', fontsize=9, color='gray')
ax.set_ylabel('Suhu Rata-rata Tahunan (°C)')
ax.set_title('Proyeksi Suhu Tahunan — Yogyakarta', fontsize=13)
ax.legend()

# --- Curah Hujan ---
ax2 = axes[1]
ax2.plot(hujan_hist_yr.index, hujan_hist_yr, color='tomato', lw=2,
         marker='o', ms=3, label='Historis (Obs)')
ax2.plot(hujan_245_yr.index, hujan_245_yr, color='steelblue', lw=2, label='GCM SSP2-4.5')
ax2.plot(hujan_585_yr.index, hujan_585_yr, color='darkorange', lw=2, label='GCM SSP5-8.5')
ax2.axvline(2020.5, color='gray', lw=1.5, linestyle='--', alpha=0.7)
ax2.set_ylabel('Curah Hujan Tahunan (mm/tahun)')
ax2.set_title('Proyeksi Curah Hujan Tahunan — Yogyakarta', fontsize=13)
ax2.legend()

plt.tight_layout()
plt.savefig('plot_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot time series tersimpan')

---
## 4. Siklus Musiman (Climatology)

In [ ]:
# ============================================================
# SIKLUS MUSIMAN BULANAN
# ============================================================
def musiman_bulanan(df, col):
    df = df.copy()
    df['month'] = df['date'].dt.month
    return df.groupby('month')[col].mean()

suhu_hist_m  = musiman_bulanan(obs, 'suhu')
suhu_245_m   = musiman_bulanan(gcm_245, 'suhu')
suhu_585_m   = musiman_bulanan(gcm_585, 'suhu')
hujan_hist_m = musiman_bulanan(obs, 'hujan') * 30  # mm/bulan (approx)
hujan_245_m  = musiman_bulanan(gcm_245, 'hujan') * 30
hujan_585_m  = musiman_bulanan(gcm_585, 'hujan') * 30

months = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Suhu musiman ---
ax = axes[0]
ax.plot(range(1,13), suhu_hist_m,  'o-', lw=2, color='tomato',    label='Historis')
ax.plot(range(1,13), suhu_245_m,   's-', lw=2, color='steelblue', label='SSP2-4.5')
ax.plot(range(1,13), suhu_585_m,   '^-', lw=2, color='darkorange', label='SSP5-8.5')
ax.fill_between(range(1,13), suhu_hist_m, suhu_585_m, alpha=0.1, color='darkorange')
ax.set_xticks(range(1,13))
ax.set_xticklabels(months, rotation=45)
ax.set_ylabel('Suhu (°C)')
ax.set_title('Siklus Musiman Suhu', fontsize=12)
ax.legend()

# --- Hujan musiman ---
ax2 = axes[1]
ax2.bar(np.array(range(1,13)) - 0.25, hujan_hist_m,  0.25, label='Historis',  color='tomato',    alpha=0.8)
ax2.bar(np.array(range(1,13)),         hujan_245_m,   0.25, label='SSP2-4.5',  color='steelblue', alpha=0.8)
ax2.bar(np.array(range(1,13)) + 0.25, hujan_585_m,   0.25, label='SSP5-8.5',  color='darkorange', alpha=0.8)
ax2.set_xticks(range(1,13))
ax2.set_xticklabels(months, rotation=45)
ax2.set_ylabel('Curah Hujan (mm/bulan)')
ax2.set_title('Siklus Musiman Curah Hujan', fontsize=12)
ax2.legend()

plt.suptitle('Siklus Musiman: Historis vs Proyeksi — Yogyakarta', fontsize=14)
plt.tight_layout()
plt.savefig('plot_musiman.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot musiman tersimpan')

---
## 5. Perbandingan Periode

In [ ]:
# ============================================================
# DEFINISI PERIODE
# Historis   : 1990–2020
# Near future: 2021–2040
# Far future : 2041–2060
# ============================================================
def filter_tahun(df, t1, t2):
    df = df.copy()
    df['year'] = df['date'].dt.year
    return df[df['year'].between(t1, t2)]

hist_data       = filter_tahun(obs,     1990, 2020)
near_245        = filter_tahun(gcm_245, 2021, 2040)
far_245         = filter_tahun(gcm_245, 2041, 2060)
near_585        = filter_tahun(gcm_585, 2021, 2040)
far_585         = filter_tahun(gcm_585, 2041, 2060)

# ============================================================
# BOXPLOT PERBANDINGAN
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

labels = ['Hist\n1990–2020', 'SSP245\n2021–2040', 'SSP245\n2041–2060',
          'SSP585\n2021–2040', 'SSP585\n2041–2060']
data_suhu  = [hist_data['suhu'],  near_245['suhu'],  far_245['suhu'],
              near_585['suhu'],   far_585['suhu']]
data_hujan = [hist_data['hujan'], near_245['hujan'], far_245['hujan'],
              near_585['hujan'],  far_585['hujan']]
colors = ['tomato', 'lightblue', 'steelblue', 'moccasin', 'darkorange']

# Suhu
bp1 = axes[0].boxplot(data_suhu, patch_artist=True, labels=labels, showfliers=False)
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[0].set_ylabel('Suhu (°C)')
axes[0].set_title('Distribusi Suhu Harian per Periode', fontsize=12)
axes[0].tick_params(axis='x', labelsize=9)

# Hujan
bp2 = axes[1].boxplot(data_hujan, patch_artist=True, labels=labels, showfliers=False)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1].set_ylabel('Curah Hujan Harian (mm)')
axes[1].set_title('Distribusi Curah Hujan Harian per Periode', fontsize=12)
axes[1].tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('plot_perbandingan.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PERUBAHAN SUHU (DELTA) TERHADAP BASELINE
# ============================================================
baseline_suhu  = hist_data['suhu'].mean()
baseline_hujan = hist_data['hujan'].mean()

print('📊 Perubahan terhadap Baseline (1990–2020):')
print(f'\n  Baseline suhu   : {baseline_suhu:.2f} °C')
print(f'  Baseline hujan  : {baseline_hujan:.2f} mm/hari')
print()

for label, df in [('SSP2-4.5 Near (2021–2040)', near_245),
                  ('SSP2-4.5 Far  (2041–2060)', far_245),
                  ('SSP5-8.5 Near (2021–2040)', near_585),
                  ('SSP5-8.5 Far  (2041–2060)', far_585)]:
    delta_t = df['suhu'].mean() - baseline_suhu
    delta_h = (df['hujan'].mean() - baseline_hujan) / baseline_hujan * 100
    print(f'  {label}:')
    print(f'    ΔSuhu  : {delta_t:+.2f} °C')
    print(f'    ΔHujan : {delta_h:+.1f} %')
    print()

---
## 6. Export CSV

In [ ]:
# Export data tahunan untuk analisis lanjutan
df_export = pd.DataFrame({
    'year'         : suhu_hist_yr.index.tolist() + suhu_245_yr.index.tolist() + suhu_585_yr.index.tolist(),
    'suhu_mean'    : suhu_hist_yr.tolist() + suhu_245_yr.tolist() + suhu_585_yr.tolist(),
    'hujan_total'  : hujan_hist_yr.tolist() + hujan_245_yr.tolist() + hujan_585_yr.tolist(),
    'skenario'     : (['Historis'] * len(suhu_hist_yr) +
                      ['SSP245']   * len(suhu_245_yr)  +
                      ['SSP585']   * len(suhu_585_yr)),
})

df_export.to_csv('data_proyeksi_tahunan.csv', index=False)
print('✅ Data tersimpan: data_proyeksi_tahunan.csv')
print(f'   Total baris: {len(df_export)}')
print(df_export.head(3))